# Hisab Pata — Qwen 1.5B Fine-tune on Kaggle

Dataset: `train.jsonl` (6832 Bengali/Banglish examples)
Model: `Qwen/Qwen1.5-1.8B`
Output: Fine-tuned model → upload to Hugging Face

In [ ]:
# Install deps
!pip install -q transformers datasets accelerate peft bitsandbytes trl
!pip install -q huggingface_hub

In [ ]:
# Upload train.jsonl to Kaggle dataset first, or upload manually:
# 1. Open Kaggle → Create New Dataset → Upload train.jsonl
# 2. Or just upload files here in the notebook (folder icon → upload)
# Then update the path below:
DATA_PATH = '/kaggle/input/hisabpata-dataset/train.jsonl'
# If you uploaded to current dir:
# DATA_PATH = 'train.jsonl'

In [ ]:
import json, torch, re, random
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print('Loading dataset...')
with open(DATA_PATH) as f:
    data = [json.loads(l) for l in f]
print(f'Examples: {len(data)}')

In [ ]:
# Convert ChatML to plain text for SFT
def format_chat(ex):
    msgs = ex['messages']
    text = ''
    for m in msgs:
        if m['role'] == 'system':
            text += f'<|system|>\n{m["content"]}\n'
        elif m['role'] == 'user':
            text += f'<|user|>\n{m["content"]}\n'
        elif m['role'] == 'assistant':
            text += f'<|assistant|>\n{m["content"]}\n'
    text += '<|end|>'
    return text

texts = [format_chat(ex) for ex in data]
dataset = Dataset.from_list([{'text': t} for t in texts])
print(f'Example:\n{texts[0][:300]}...')

In [ ]:
MODEL_NAME = 'Qwen/Qwen1.5-1.8B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)
model.config.use_cache = False
print('Model loaded')

In [ ]:
# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

# Training args
training_args = TrainingArguments(
    output_dir='/kaggle/working/hisabpai-qwen',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
    bf16=True,
    max_grad_norm=0.3
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    formatting_func=lambda x: x['text'],
    max_seq_length=512,
    peft_config=lora_config
)
print('Trainer ready')

In [ ]:
# TRAIN START
trainer.train()
print('Training complete')

In [ ]:
# Save LoRA adapters
output_path = '/kaggle/working/hisabpai-qwen-final'
trainer.model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print(f'Saved to {output_path}')

# Zip for download
!zip -r /kaggle/working/hisabpai-qwen-final.zip /kaggle/working/hisabpai-qwen-final
print('Zip ready')

In [ ]:
# Quick test: run inference
from peft import PeftModel

# Reload base + LoRA
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map='auto', trust_remote_code=True,
    torch_dtype=torch.bfloat16
)
model_lora = PeftModel.from_pretrained(base, output_path)

test_inputs = [
    '500 টাকা খরচ করেছি পরিবহন বাবদ',
    'কে তুমি?',
    'ব্যালেন্স কত?',
]

for inp in test_inputs:
    prompt = f"""<|system|>
book_type: personal
categories: Transport, Mobile Recharge, Postage, Publication, Office Stationery, Tips, Donation, Others, Salary
balance: 12500
""" + f"""
<|user|>
{inp}
<|assistant|>
"""
    tokens = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model_lora.generate(**tokens, max_new_tokens=128, do_sample=False)
    response = tokenizer.decode(out[0], skip_special_tokens=False)
    print(f'\nQ: {inp}')
    print(f'A: {response}')

In [ ]:
# (Optional) Upload to Hugging Face
# from huggingface_hub import notebook_login
# notebook_login()
# model_lora.push_to_hub('your-username/hisabpai-qwen')
# tokenizer.push_to_hub('your-username/hisabpai-qwen')